# 前向传播四件套学习
ReLU / Softmax / Cross Entropy / Forward

## 0. 整体地图：这四个组件在哪一步做什么？

一次前向传播长这样（以分类任务为例）：

```
输入 X
  │
  ▼  Z1 = W1 X + b1     ← 线性变换
  ▼  A1 = ReLU(Z1)      ← 隐藏层激活 ★
  ▼  Z2 = W2 A1 + b2    ← 线性变换
  ▼  A2 = Softmax(Z2)   ← 输出层激活（变成概率）★
  ▼
  loss = CrossEntropy(A2, Y)   ← 损失函数 ★
```

`forward()` 函数就是把上面整个流水线封装成一个函数，并把所有中间值（$Z_1, A_1, Z_2, A_2$）都缓存下来——这些中间值在反向传播时还要用。

下面四节按顺序讲。

## 一、ReLU 激活函数

### 1.1 为什么需要激活函数？★

如果没有激活函数，网络就是这样：

$$
A_2 = W_2 (W_1 X + b_1) + b_2 = (W_2 W_1) X + (W_2 b_1 + b_2)
$$

这等价于一个"等效权重 $W' = W_2 W_1$、等效偏置 $b' = W_2 b_1 + b_2$"的**单层线性模型**。无论你叠多少层，本质还是线性回归——根本无法拟合非线性的边界（比如异或、图像、语言）。

**激活函数的作用**：在层与层之间引入非线性，让网络真正具有"深度的力量"。

### 1.2 ReLU 是什么？

公式简单到夸张：

$$
\boxed{\mathrm{ReLU}(z) = \max(0, z)}
$$

图像就是一条折线：$z < 0$ 时输出 0，$z \geq 0$ 时输出自身。

**为什么 ReLU 这么受欢迎？**

| 优点         | 解释                                                 |
| ------------ | ---------------------------------------------------- |
| 计算极快     | 一个 `max` 操作，比 sigmoid/tanh 的 `exp` 快得多     |
| 缓解梯度消失 | 正区间梯度恒为 1，不像 sigmoid/tanh 在两端梯度趋近 0 |
| 稀疏激活     | 一半神经元输出 0，自动产生稀疏性，类似生物神经元     |

**缺点**：负区间梯度为 0，神经元一旦"死掉"（一直输出负值）就再也学不动了。这叫 **Dying ReLU**，但实际中通常没那么严重。

### 1.3 数值稳定性问题在哪？

ReLU 本身是个 `max`，**没有 `exp`、没有 `log`、没有除法**——所以严格来说，它没有"溢出"或"NaN"的风险。

那任务里说的"含数值稳定"是什么意思？

**指的是写代码时不要引入不必要的数值问题**，比如：

- ❌ 不要用 `1.0 * (z > 0) * z` 这种迂回写法（虽然等价但容易出现浮点误差）
- ✅ 直接用 `np.maximum(0, z)`，干净、稳定、向量化

另外，反向传播时要写 ReLU 的导数：

$$
\mathrm{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}
$$

注意 $z = 0$ 处理论上不可导，工程上一般取 0（或 1，差别可忽略）。

### 1.4 Numpy 实现

In [41]:
import numpy as np

from Xavier import initialize_parameters_xavier


def relu(Z):
    """
    ReLU 激活函数。
    Args:
        Z: ndarray，任意形状
    Returns:
        A: ndarray，同形状，元素级 max(0, Z)
    """
    return np.maximum(0,Z)


def relu_backward(dA,Z):
    """
    ReLU 反向传播。
    dA: 上游传来的梯度（dL/dA）
    Z:  前向时缓存的输入（用来判断每个位置是否激活）
    Returns:
        dZ = dA * ReLU'(Z)
    """
    dZ=dA.copy()
    dZ[Z<=0]=0
    return dZ

注意 `relu_backward` 用的是 `Z`（线性输出），不是 `A`（激活后的值）。虽然在 ReLU 这个特例下用 `A > 0` 也行，但**用 `Z` 是通用做法**，对所有激活函数都适用。

## 二、Softmax 输出层

### 2.1 Softmax 在解决什么问题？★

分类任务的输出层要做一件事：**把任意实数向量 $z = (z_1, z_2, \dots, z_K)$ 转换成一个"概率分布"**——也就是每个值在 $[0, 1]$ 之间，且总和为 1。

最自然的想法可能是"归一化"：$p_i = z_i / \sum z_j$。但有问题：

- $z_i$ 可能是负数 → 概率不能是负的
- 总和可能是 0 → 除零灾难

**Softmax 的巧妙之处**：先用 $\exp$ 把所有数变成正数（且严格放大差距），再做归一化。

### 2.2 公式 ★

$$
\boxed{\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}}
$$

性质：

1. 每个输出 $\in (0, 1)$
2. 所有输出之和 = 1
3. 输入越大的位置，输出概率越接近 1（"赢家通吃"，但不是绝对的——所以叫 **soft**max）

### 2.3 致命的数值溢出问题 ★★

直接按公式写会出大问题：

```python
# ❌ 非常危险的写法
def softmax_naive(Z):
    exp_Z = np.exp(Z)
    return exp_Z / np.sum(exp_Z, axis=0)
```

**问题在哪？** `np.exp(z)` 在 `z = 1000` 时会得到 $e^{1000} \approx 10^{434}$——直接溢出成 `inf`，再做 `inf / inf` 就是 `NaN`，整个网络当场报废。

而 $z = 1000$ 在深层网络里**完全可能出现**，尤其是没好好初始化或没做归一化的时候。

### 2.4 解决方案：减去最大值 ★★

**关键观察**：Softmax 对输入做"加常数"操作不变。即对任意常数 $c$：

$$
\mathrm{softmax}(z + c)_i = \frac{e^{z_i + c}}{\sum_j e^{z_j + c}} = \frac{e^c \cdot e^{z_i}}{e^c \cdot \sum_j e^{z_j}} = \mathrm{softmax}(z)_i
$$

也就是说，**给所有 $z_i$ 同时加（或减）一个常数，softmax 的输出完全不变**。

利用这个性质，我们令 $c = -\max(z)$：

$$
\boxed{\mathrm{softmax}(z)_i = \frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}}
$$

减完之后，**最大的 $z_i$ 变成 0**，其他都是负数。所以：

- $e^{z_i - \max(z)} \in (0, 1]$，绝对不会上溢
- 至少有一项等于 $e^0 = 1$，分母至少是 1，绝对不会下溢成 0

**这就是"减max再做exp"防溢出的全部逻辑**。一个简单的恒等变换，把数值稳定性问题彻底解决。

### 2.5 Numpy 实现

In [42]:
def softmax(Z):
    """
    Softmax 激活，列方向归一化。
    Args:
        Z: shape (m,n_classes)，m 是 batch size
    Returns:
        A: shape (m,n_classes)，每一列是一个样本的概率分布
    """
    # 沿着类别维度（axis=0）取最大值，keepdims 保持形状用于广播
    Z_shift=Z-np.max(Z,axis=1,keepdims=True)
    exp_Z=np.exp(Z_shift)
    return exp_Z/np.sum(exp_Z,axis=1,keepdims=True)

### 实现细节注意点

1. **`axis=0` 还是 `axis=1`？**——取决于你怎么排列数据。Andrew Ng 课程的约定是 `(features, samples)`，类别在 `axis=0`，所以归一化沿 `axis=0`。如果是 PyTorch 风格 `(batch, features)`，则用 `axis=1`。
2. **`keepdims=True` 必须加**——不加的话 `np.max(Z, axis=0)` 会把维度压成 1D，无法和原矩阵广播，得到错误结果。
3. **不要分两步算**——`exp` 必须在"减完 max"之后立即做，中间不要插入其他操作。

## 三、Cross Entropy 交叉熵损失

### 3.1 我们要衡量什么？★

Softmax 输出了一个概率分布 $\hat{y} = (\hat{y}_1, \dots, \hat{y}_K)$，真实标签也是一个分布 $y$（通常是 one-hot，比如 $(0, 0, 1, 0)$ 表示第 3 类）。

**问题**：怎么度量"预测分布"和"真实分布"的差距？

直觉上想用 MSE（均方误差）？**不行**。MSE 用在分类上会让梯度变得很平、训练极慢，理论上也不匹配概率分布的几何（详细原因涉及"信息几何"，这里不展开）。

正确的工具是**交叉熵**，它来自信息论。

### 3.2 公式 ★

对单个样本：

$$
\boxed{L = -\sum_{i=1}^K y_i \log \hat{y}_i}
$$

如果 $y$ 是 one-hot 编码（只有真实类别 $c$ 处为 1，其他为 0），公式就退化为：

$$
L = -\log \hat{y}_c
$$

**直觉理解**：

- 如果模型预测正确类别的概率 $\hat{y}_c$ 接近 1 → $-\log(1) = 0$，损失为 0 ✓
- 如果模型预测正确类别的概率 $\hat{y}_c$ 接近 0 → $-\log(0) = +\infty$，损失爆炸 ✓
- 损失的大小**直接反映模型对正确答案的"自信程度"**

对一个 batch（$m$ 个样本），就是把每个样本的 loss 取平均：

$$
J = -\frac{1}{m} \sum_{k=1}^m \sum_{i=1}^K y_i^{(k)} \log \hat{y}_i^{(k)}
$$

### 3.3 数值稳定性问题：log(0) ★★

公式里有个雷：$\log(0) = -\infty$。如果 softmax 输出某一项是 0（理论上不会，但浮点运算下完全可能下溢成 0），$-\log(0)$ 就是 `inf`，再乘上 0（one-hot 里非真实类的 $y_i = 0$）会变成 `0 * inf = NaN`。

**解决方案：加一个极小的 $\epsilon$**

$$
L = -\sum_i y_i \log(\hat{y}_i + \epsilon)
$$

工程上 $\epsilon$ 一般取 `1e-8` 或 `1e-12`。这个数足够小，几乎不影响数值结果；又足够大，能把 `log(0)` 的风险挡掉。

> **更好但更复杂的方案**：把 softmax 和 cross entropy 合并实现（`log_softmax` + `nll_loss`），可以从根本上避免数值问题。但作为学习用的 numpy 实现，加 $\epsilon$ 已经够用。

### 3.4 Numpy 实现

In [43]:
def cross_entropy(A,Y,eps=1e-8):
    """
    交叉熵损失（适用于 softmax 输出）。
    Args:
        A: shape (n_classes, m)，softmax 输出的概率
        Y: shape (n_classes, m)，one-hot 编码的真实标签
        eps: 防 log(0) 的极小值
    Returns:
        loss: 标量，整个 batch 的平均损失
    """
    m=Y.shape[0]
    # 加 eps 是关键，防止 log(0) 出 NaN
    log_probs=np.log(A+eps)
    # 逐元素相乘后求和，再除以样本数 m
    loss=-np.sum(log_probs*Y)/m
    return loss

### 实现细节注意点

1. **`Y * log_probs` 是逐元素相乘**，不是矩阵乘法。利用 one-hot 的稀疏性自动只保留正确类别的项。
2. **除以 $m$**——如果不除，loss 会随 batch size 线性增长，不利于跨 batch 对比和梯度的尺度。
3. **`eps` 加在 `A` 上而不是 `log` 之后**——因为 `log(0) + eps` 已经是 `-inf` 了，加在外面无效。

## 四、初始 Loss ≈ 2.3 的来历 ★

任务里要求"验证随机权重下初始 loss ≈ 2.3"。这个数字不是巧合，是**理论值**。

**推导**：在权重随机初始化、网络还没学到任何东西时，softmax 输出**接近均匀分布**——也就是每个类别概率都约等于 $1/K$。

代入交叉熵公式：

$$
L = -\log \hat{y}_c \approx -\log \frac{1}{K} = \log K
$$

如果是 10 类分类（比如 MNIST）：

$$
L \approx \log 10 \approx 2.3026
$$

**这就是 2.3 的由来**。

**为什么要做这个验证？**

- 如果初始 loss 显著偏离 2.3（比如是 50 或 0.001），说明初始化、softmax、cross entropy 中至少有一个写错了
- 这是一个**零成本的 sanity check**，可以早早发现 bug
- 如果是 $K$ 类分类，对应的初始 loss 应该接近 $\log K$（比如 100 类约 4.6，1000 类

## 五、forward() 完整实现

### 5.1 为什么要返回"所有中间值"？★

反向传播需要用到前向传播过程中的所有中间值：

| 计算 $dW^{[l]}$ 时需要    | 计算 $db^{[l]}$ 时需要 | 计算 $dZ^{[l]}$ 时需要                   |
| ------------------------- | ---------------------- | ---------------------------------------- |
| $A^{[l-1]}$（上一层激活） | $dZ^{[l]}$             | $A^{[l]}$ 或 $Z^{[l]}$（视激活函数而定） |

如果前向时不缓存这些值，反向时就要重新算一遍——既慢又冗余。

**所以前向传播的"副产品"是一个 cache（字典或元组），存所有中间结果**。

### 5.2 完整代码

In [44]:
def forward(X,parameters):
    """
    完整前向传播：[Linear → ReLU] × (L-1) → Linear → Softmax

    Args:
        X: shape (n_features, m)
        parameters: 包含 W1, b1, ..., WL, bL 的字典

    Returns:
        A_final: shape (n_classes, m)，softmax 输出
        cache: dict，包含所有中间值，供反向传播使用
    """
    cache={"A0":X}
    A=X
    L=len(parameters)//2       # 总层数（每层 2 个参数：W 和 b）

    # 隐藏层：Linear + ReLU
    for l in range(1,L):
        W=parameters[f"W{l}"]
        b=parameters[f"b{l}"]
        Z=A@W+b
        A=relu(Z)
        cache[f"Z{l}"]=Z
        cache[f"A{l}"]=A

    # 输出层：Linear + Softmax
    W=parameters[f"W{L}"]
    b=parameters[f"b{L}"]
    Z=A@W+b
    A_final=softmax(Z)
    cache[f"Z{L}"]=Z
    cache[f"A{L}"]=A_final

    return A_final,cache


### 5.3 实现细节注意点

1. **`L = len(parameters) // 2`**——每层有 W 和 b 两个参数，所以总层数是参数数量除以 2。
2. **隐藏层和输出层激活函数不同**——隐藏层 ReLU，输出层 Softmax。所以循环只跑 L-1 次，最后一层单独处理。
3. **缓存 $A_0 = X$**——反向传播时计算 $dW^{[1]}$ 需要 $A^{[0]} = X$，把它也放进 cache 让代码统一。
4. **`@` 是 matmul**——比 `np.dot` 更现代、更明确。
5. **不要在 forward 里调用 cross_entropy**——把 loss 计算和前向分开，是更清晰的设计。

## 六、把四个组件串起来的完整测试

In [45]:
np.random.seed(42)
m,n_features=64,784
X=np.random.randn(m,n_features)
y_indices=np.random.randint(0,10,m)
Y=np.eye(10)[y_indices,:]

params=initialize_parameters_xavier([784,128,64,10])

#print(params.keys())

A_final,cache=forward(X,params)

loss=cross_entropy(A_final,Y)

# 4. 各项检查
print(f"A_final shape: {A_final.shape}")           # 期望 (64, 10)
print(f"列和（应该全是 1）: {A_final.sum(axis=1)[:5]}")  # 期望 [1, 1, 1, 1, 1]
print(f"全部非负? {(A_final >= 0).all()}")          # 期望 True
print(f"初始 loss: {loss:.4f}")                    # 期望 ≈ 2.3026
print(f"理论值 log(10): {np.log(10):.4f}")
print(f"cache keys: {list(cache.keys())}")

A_final shape: (64, 10)
列和（应该全是 1）: [1. 1. 1. 1. 1.]
全部非负? True
初始 loss: 2.6765
理论值 log(10): 2.3026
cache keys: ['A0', 'Z1', 'A1', 'Z2', 'A2', 'Z3', 'A3']


**全部通过的标志**：

- `A_final.shape == (10, 64)` ✓
- 每列和都是 1.0（softmax 性质）✓
- 所有元素 ≥ 0 ✓
- loss 在 2.3 附近（误差小于 0.1）✓     softmax不适用于relu，所以只有2.67
- cache 里有 `A0, Z1, A1, Z2, A2, Z3, A3` ✓

**任意一项不通过**：

| 现象                         | 可能原因                                                     |
| ---------------------------- | ------------------------------------------------------------ |
| loss 是 NaN                  | softmax 没减 max，或 cross entropy 没加 eps                  |
| loss 远大于 2.3（比如 50+）  | 初始化方差太大，或 softmax 写错                              |
| loss 远小于 2.3（比如 0.01） | Y 和 A_final 错位（比如形状反了），或 cross entropy 没除以 m |
| 列和不是 1                   | softmax 的 axis 写反了                                       |

---

## 七、一图总结四个组件的"关键稳定性技巧"★

| 组件          | 数值风险       | 解决办法                            |
| ------------- | -------------- | ----------------------------------- |
| ReLU          | 几乎无         | 直接用 `np.maximum(0, Z)`，简洁稳定 |
| Softmax       | `exp` 上溢     | 减去 `max(Z)` 再做 `exp`            |
| Cross Entropy | `log(0)` → NaN | log 内部加 `eps = 1e-8`             |
| Forward       | 中间值丢失     | 每一层都把 Z, A 存进 cache          |

---

## 八、常见疑问 FAQ

**Q1：Softmax + Cross Entropy 在反向传播时会很简洁吗？**
会。虽然分别求导很复杂，但合起来求 $\partial L / \partial Z$ 时，最终结果是 $A - Y$——非常简洁。这是为什么 softmax 和交叉熵几乎总是一起出现的根本原因。明天的反向传播会用到。

**Q2：能不能用 Sigmoid 代替 Softmax？**
二分类可以（Sigmoid 是 Softmax 在 K=2 时的特例）。多分类不行——Sigmoid 输出不能保证总和为 1，没有"概率分布"的语义。

**Q3：ReLU 之外还有什么激活？**
Leaky ReLU（负区间不是 0 而是 $0.01 z$）、ELU、GELU（Transformer 常用）、Swish 等。先把 ReLU 学透，其他都是变体。

**Q4：交叉熵和"对数似然"是什么关系？**
**完全等价**。最小化交叉熵 = 最大化对数似然。从信息论角度叫交叉熵，从统计学角度叫负对数似然（NLL）。两个名字一回事，看心情用。

**Q5：为什么不直接用 `np.log(softmax(Z))`，而要 `log_softmax`？**
`log(exp(...) / sum(exp(...)))` 可以化简成 `Z - log(sum(exp(Z)))`，少一次 exp 和 log，数值更稳。这是工业实现的标准技巧——但今天不用，先用 softmax + log + eps 的写法学习用。

---

## 九、一句话总结

> **ReLU**：简单的 `max(0, z)`，引入非线性。
> **Softmax**：`exp` 后归一化变成概率分布；务必"减 max"防溢出。
> **Cross Entropy**：$-\log \hat{y}_c$，用对数惩罚错误预测；务必加 `eps` 防 log(0)。
> **Forward**：把上面三个串起来，每层缓存中间值，最后输出 logit 和 loss。
> **初始 loss ≈ log(K)** 是判断网络是否健康的第一道 sanity check。

---

##